# PyCAM-SIMA live-field Notebook

This Notebook starts one controller plus 24 CAM-SIMA MPI workers, advances the model one step at a time, and returns live NumPy fields to this kernel.

It works from a Derecho login-node Jupyter kernel: `model.start()` automatically submits a 24-rank PBS worker and waits for it to connect. Inside an existing compute allocation it launches locally. If `pycam_sima` is not installed in the selected kernel, run `%pip install -e /glade/work/ruitong/pycam-sima` once and restart the kernel.

In [14]:
from datetime import datetime
from pathlib import Path
import os
import shutil

from pycam_sima import NotebookSession
import pycam_sima

repo = Path("/glade/work/ruitong/pycam-sima")
case = repo / "reference/cases/FKESSLER_ne3pg3_gnu_24x50"
reference_run = Path(
    "/glade/derecho/scratch/ruitong/pycam-sima/"
    "FKESSLER_ne3pg3_gnu_24x50/FKESSLER_ne3pg3_gnu_24x50/run"
)
scratch = Path(os.environ.get("SCRATCH", "/glade/derecho/scratch/ruitong"))
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
run_dir = scratch / "pycam-sima/notebook_trials" / stamp / "run"
run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_run / "atm_in", run_dir / "atm_in")
print(run_dir)
print(pycam_sima.__version__)

/glade/derecho/scratch/ruitong/pycam-sima/notebook_trials/20260718-184028/run
0.3.0


## Start the MPI model

This cell returns only after all 24 ranks have completed CAM initialization. Re-running it first closes an older session. If that session already produced history files, run the setup cell again to create a fresh directory.

In [15]:
if "model" in globals() and model.running:
    model.close()

existing_history = tuple(run_dir.glob("*.cam.h*.nc"))
if existing_history:
    raise RuntimeError("run the setup cell again to create a fresh run directory")

model = NotebookSession(
    repo / "configs/fkessler_ne3pg3.yaml",
    run_dir=run_dir,
    env_script=case / ".env_mach_specific.sh",
    python_executable=repo / ".venv/bin/python",
    log_path=run_dir / "mpi-worker.log",
)
model.start()
print(
    f"ready: mode={model.launch_mode_used}, job={model.job_id}, "
    f"ranks={model.ranks}, fields={len(model.field_names)}, step={model.current_step}"
)

PyCAM-SIMA PBS worker submitted as 6789816.desched1; waiting for 24 MPI ranks ...
ready: mode=pbs, job=6789816.desched1, ranks=24, fields=21, step=0


## Inspect available fields

In [16]:
model.field_names

('air_temperature',
 'eastward_wind',
 'northward_wind',
 'surface_air_pressure',
 'air_pressure_thickness',
 'air_pressure_thickness_of_dry_air',
 'air_pressure',
 'air_pressure_of_dry_air',
 'air_pressure_at_interface',
 'air_pressure_of_dry_air_at_interface',
 'surface_pressure_of_dry_air',
 'surface_geopotential',
 'geopotential_height_wrt_surface',
 'geopotential_height_wrt_surface_at_interface',
 'lagrangian_tendency_of_air_pressure',
 'reciprocal_of_dimensionless_exner_function_wrt_surface_air_pressure',
 'dry_static_energy',
 'tendency_of_air_temperature_due_to_model_physics',
 'tendency_of_eastward_wind_due_to_model_physics',
 'tendency_of_northward_wind_due_to_model_physics',
 'ccpp_constituents')

## Read a field before stepping

`get_field` returns a copied rank-local NumPy array. Here `temperature` remains directly available to later Notebook cells.

In [17]:
field_name = "air_temperature"
rank = 0
temperature = model.get_field(field_name, rank=rank)
stats = model.get_field_stats(field_name, rank=rank)
print(stats)
temperature

{'rank': 0, 'shape': (27, 30), 'dtype': '<f8', 'min': 149.8407866754426, 'max': 306.29054325059553, 'mean': 237.51537148398492}


array([[150.20253396, 158.94337244, 167.16118088, 174.73367559,
        181.57244239, 187.71430852, 193.05934551, 197.28293981,
        201.03648614, 205.10012764, 209.52062187, 214.33856299,
        219.59632439, 225.33312797, 231.5994718 , 238.41764954,
        245.79321125, 253.68209419, 261.98531497, 269.87826244,
        276.48345972, 281.6156288 , 285.31699146, 287.64103729,
        289.22459305, 290.65616286, 291.93823274, 293.07193045,
        294.05730233, 294.89447155],
       [150.32655508, 159.19389267, 167.58283913, 175.34162418,
        182.34181255, 188.59546082, 193.99676679, 198.23268511,
        201.97220267, 205.99395396, 210.33777076, 215.03799062,
        220.13024923, 225.64641816, 231.62669153, 238.08725527,
        245.02996247, 252.4160406 , 260.16325502, 267.52475172,
        273.70461795, 278.53314474, 282.03501367, 284.24340996,
        285.75205094, 287.11841745, 288.34391967, 289.42887324,
        290.37274642, 291.17523502],
       [150.42552706, 159.3942

## Advance exactly one requested step and read the new value

In [18]:
step = model.step()
temperature = model.get_field(field_name, rank=rank)
stats = model.get_field_stats(field_name, rank=rank)
print(f"step={step}, T[0,0]={temperature[0, 0]:.17g}")
print(stats)
temperature

step=1, T[0,0]=150.20534543730321
{'rank': 0, 'shape': (27, 30), 'dtype': '<f8', 'min': 149.84036762606746, 'max': 306.30091247707594, 'mean': 237.5147291811104}


array([[150.20534544, 158.94713734, 167.16574254, 174.73885624,
        181.57847386, 187.72100453, 193.06647075, 197.29024932,
        201.04385377, 205.10759605, 209.52762345, 214.34502855,
        219.60208291, 225.3376915 , 231.60227458, 238.41807997,
        245.79067885, 253.67614578, 261.97577266, 269.86567683,
        276.46880026, 281.59974335, 285.30046683, 287.62417828,
        289.20752479, 290.638921  , 291.92084469, 293.05441746,
        294.03968189, 294.87688868],
       [150.32883535, 159.19681594, 167.586263  , 175.34536503,
        182.3460191 , 188.59992459, 194.00126521, 198.23704806,
        201.9763285 , 205.99667928, 210.34090137, 215.04027286,
        220.13129697, 225.64553904, 231.62344136, 238.08122346,
        245.02085067, 252.40374813, 260.14793931, 267.50712076,
        273.68549993, 278.51316367, 282.01452839, 284.22268681,
        285.73115878, 287.09738417, 288.32276991, 289.40762682,
        290.35141991, 291.15399004],
       [150.42667709, 159.3954

## Continue step by step

Change `additional_steps` as needed. Each iteration obtains the field from the live MPI model, not from an NPZ file.

In [19]:
additional_steps = 4
for _ in range(additional_steps):
    step = model.step()
    temperature = model.get_field(field_name, rank=rank)
    stats = model.get_field_stats(field_name, rank=rank)
    print(
        f"step={step} T[0,0]={temperature[0, 0]:.17g} "
        f"min={stats['min']:.17g} max={stats['max']:.17g}"
    )

step=2 T[0,0]=150.20693921066984 min=149.84058007737622 max=306.30448088433747
step=3 T[0,0]=150.20820601916299 min=149.84156638204053 max=306.30900914520896
step=4 T[0,0]=150.20924224400252 min=149.84335775755804 max=306.31525507341388
step=5 T[0,0]=150.21008822330455 min=149.84449653997444 max=306.3227371844149


## Optional: modify a live field

Running the next cell changes rank-zero CAM memory and intentionally breaks BFB. It is left commented out by default.

In [20]:
# changed = model.get_field("air_temperature", rank=0)
# changed[0, 0] += 0.01
# model.set_field("air_temperature", changed, rank=0)
# model.step()

## Finalize

Always run this cell when finished so CAM finalizes and all MPI worker processes exit.

In [21]:
model.close()
print(f"closed; output directory: {run_dir}")

closed; output directory: /glade/derecho/scratch/ruitong/pycam-sima/notebook_trials/20260718-184028/run


## Check the available steps for BFB

Run this after `model.close()`. It compares every history timestamp produced by this Notebook with the matching native CAM-SIMA reference timestamp. It works after one step or any partial run; later reference timestamps are intentionally ignored. The five prognostic fields are compared element by element with exact dtype, shape, and value equality. Running the optional field-modification cell above should make this check report `BFB: False`.

In [22]:
from netCDF4 import Dataset
import numpy as np

bfb_fields = ("T", "Q", "U", "V", "PS")

def index_history(directory):
    indexed = {}
    for path in Path(directory).glob("*.cam.h1i.*.nc"):
        timestamp = path.name.split(".h1i.", 1)[1]
        indexed[timestamp] = path
    return indexed

reference_history = index_history(reference_run)
candidate_history = index_history(run_dir)
if not candidate_history:
    raise RuntimeError(f"no CAM history files found in {run_dir}")

first_difference = None
compared_fields = 0
for timestamp, candidate_file in sorted(candidate_history.items()):
    reference_file = reference_history.get(timestamp)
    if reference_file is None:
        first_difference = {"timestamp": timestamp, "reason": "reference timestamp missing"}
        break
    with Dataset(reference_file) as reference, Dataset(candidate_file) as candidate:
        for name in bfb_fields:
            left = np.asarray(reference[name][:])
            right = np.asarray(candidate[name][:])
            if left.shape != right.shape or left.dtype != right.dtype:
                first_difference = {
                    "timestamp": timestamp,
                    "field": name,
                    "reference_shape": left.shape,
                    "candidate_shape": right.shape,
                    "reference_dtype": left.dtype.str,
                    "candidate_dtype": right.dtype.str,
                }
                break
            if not np.array_equal(left, right):
                delta = np.abs(left - right)
                first_difference = {
                    "timestamp": timestamp,
                    "field": name,
                    "differing_values": int(np.count_nonzero(left != right)),
                    "max_abs_difference": float(np.max(delta)),
                }
                break
            compared_fields += 1
    if first_difference is not None:
        break

bfb = first_difference is None
print(f"BFB: {bfb}")
print(f"compared timestamps: {len(candidate_history)}")
print(f"exact field comparisons: {compared_fields}")
print(f"fields: {bfb_fields}")
print(f"first difference: {first_difference}")

BFB: True
compared timestamps: 6
exact field comparisons: 30
fields: ('T', 'Q', 'U', 'V', 'PS')
first difference: None
